### Data Processing

In [2]:
import json
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")
EOS = tok.eos_token_id

recs = [json.loads(line) for line in open("/home/aamer/smol_llm/data/stage00/sft_train.jsonl")]
print("keys:", list(recs[0]))

r = recs[0]
p = tok(r["prompt"])["input_ids"]
c = tok(r["completion"], add_special_tokens=False)["input_ids"] + [EOS]

ids = p + c
labels = [-100] * len(p) + c

print("prompt tokens:", len(p), " completion tokens:", len(c))
print(repr(tok.decode([t for t in labels if t != -100])))

keys: ['nct_id', 'prompt', 'completion', 'document', 'gold']
prompt tokens: 796  completion tokens: 50
'{"conditions": [], "interventions": ["placemats promoting healthier featured kids\' meals", "frequent diner card"], "intervention_types": ["BEHAVIORAL"], "allocation": "RANDOMIZED"}<|endoftext|>'


In [4]:
import random

def encode(r):
    p = tok(r["prompt"])["input_ids"]
    c = tok(r["completion"], add_special_tokens=False)["input_ids"] + [EOS]
    if len(p) + len(c) > 2048:
        return None
    return {"ids": p + c, "labels": [-100] * len(p) + c}

enc = [e for e in (encode(r) for r in recs) if e is not None]
skipped = len(recs) - len(enc)

random.seed(20260906)
random.shuffle(enc)
n_val = int(len(enc) * 0.05)
val, train = enc[:n_val], enc[n_val:]

print(f"train {len(train)}  val {len(val)}  skipped {skipped}")

train 1891  val 99  skipped 0


In [5]:
import torch

def collate(batch):
    L = max(len(x["ids"]) for x in batch)
    ids = torch.full((len(batch), L), EOS)
    labels = torch.full((len(batch), L), -100)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, x in enumerate(batch):
        n = len(x["ids"])
        ids[i, :n] = torch.tensor(x["ids"])
        labels[i, :n] = torch.tensor(x["labels"])
        att[i, :n] = 1
    return ids, att, labels

ids, att, labels = collate(train[:4])
print(ids.shape, att.sum(1).tolist(), (labels != -100).sum(1).tolist())

torch.Size([4, 897]) [831, 741, 670, 897] [49, 66, 35, 42]


## LoRA


In [6]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import os

BASE = os.path.expanduser("~/smol_llm/models/smolcpt-e1-hf")   # later: use CPT trained model

model = AutoModelForCausalLM.from_pretrained(os.path.expanduser(BASE), dtype=torch.float32).to("cuda")
model = get_peft_model(model, LoraConfig(
    r=64, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1083.64it/s]


trainable params: 34,734,080 || all params: 396,555,200 || trainable%: 8.7590


In [7]:
import math
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

MICRO_BATCH, GRAD_ACCUM, EPOCHS = 4, 4, 2          # effective batch 16
steps_per_epoch = math.ceil(len(train) / (MICRO_BATCH * GRAD_ACCUM))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(0.05 * total_steps)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(params, lr=2e-4, weight_decay=0.0)
lr_scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"steps/epoch {steps_per_epoch}  total {total_steps}  warmup {warmup_steps}")

steps/epoch 119  total 238  warmup 11


In [8]:
@torch.no_grad()
def val_loss():
    model.eval()
    total, n = 0.0, 0
    for i in range(0, len(val), MICRO_BATCH):
        ids, att, labels = collate(val[i:i + MICRO_BATCH])
        ids, att, labels = ids.cuda(), att.cuda(), labels.cuda()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = model(input_ids=ids, attention_mask=att, labels=labels).loss
        k = (labels[:, 1:] != -100).sum().item()   # tokens actually scored
        total += loss.item() * k
        n += k
    model.train()
    return total / n

v0 = val_loss()
print(f"val loss before training: {v0:.4f}")

val loss before training: 1.0328


In [9]:
import time

model.train()
torch.cuda.reset_peak_memory_stats()
t0, step = time.time(), 0

for epoch in range(EPOCHS):
    random.shuffle(train)
    n_micro = math.ceil(len(train) / MICRO_BATCH)
    for m in range(n_micro):
        ids, att, labels = collate(train[m * MICRO_BATCH:(m + 1) * MICRO_BATCH])
        ids, att, labels = ids.cuda(), att.cuda(), labels.cuda()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = model(input_ids=ids, attention_mask=att, labels=labels).loss
        (loss / GRAD_ACCUM).backward()

        last = (m == n_micro - 1)
        if (m + 1) % GRAD_ACCUM == 0 or last:
            gn = torch.nn.utils.clip_grad_norm_(params, 1.0).item()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            if step % 10 == 0:
                print(f"step {step}  loss {loss.item():.4f}  grad_norm {gn:.3f}  lr {lr_scheduler.get_last_lr()[0]:.2e}")

    print(f"epoch {epoch + 1}  val loss {val_loss():.4f}")

print(f"steps {step}  wall {time.time() - t0:.0f}s  peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")

step 10  loss 1.2131  grad_norm 0.150  lr 1.82e-04
step 20  loss 0.5525  grad_norm 0.105  lr 1.99e-04
step 30  loss 0.4008  grad_norm 0.069  lr 1.97e-04
step 40  loss 0.3084  grad_norm 0.076  lr 1.92e-04
step 50  loss 0.2499  grad_norm 0.078  lr 1.86e-04
step 60  loss 0.2149  grad_norm 0.072  lr 1.78e-04
step 70  loss 0.2055  grad_norm 0.060  lr 1.68e-04
step 80  loss 0.2782  grad_norm 0.074  lr 1.58e-04
step 90  loss 0.2012  grad_norm 0.078  lr 1.46e-04
step 100  loss 0.1918  grad_norm 0.082  lr 1.33e-04
step 110  loss 0.1951  grad_norm 0.114  lr 1.20e-04
epoch 1  val loss 0.2206
step 120  loss 0.1784  grad_norm 0.073  lr 1.06e-04
step 130  loss 0.1836  grad_norm 0.069  lr 9.24e-05
step 140  loss 0.0895  grad_norm 0.079  lr 7.87e-05
step 150  loss 0.2176  grad_norm 0.091  lr 6.54e-05
step 160  loss 0.2035  grad_norm 0.071  lr 5.28e-05
step 170  loss 0.2488  grad_norm 0.095  lr 4.11e-05
step 180  loss 0.3808  grad_norm 0.093  lr 3.05e-05
step 190  loss 0.2071  grad_norm 0.074  lr 2.13e

In [10]:
model.eval()
for x in val[:5]:
    prompt_ids = [t for t, l in zip(x["ids"], x["labels"]) if l == -100]
    inp = torch.tensor([prompt_ids], device="cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        out = model.generate(inp, attention_mask=torch.ones_like(inp),
                             max_new_tokens=256, do_sample=False,
                             eos_token_id=EOS, pad_token_id=EOS)
    gen = out[0, len(prompt_ids):].tolist()
    print(f"--- stopped={EOS in gen}  tokens={len(gen)}")
    print(tok.decode(gen, skip_special_tokens=True))

--- stopped=True  tokens=51
{"conditions": ["Temporomandibular Joint Syndrome"], "interventions": ["Radial Shock Wave Therapy", "Manual Therapy"], "intervention_types": ["PROCEDURE"], "allocation": "RANDOMIZED"}
--- stopped=True  tokens=48
{"conditions": ["septic shock"], "interventions": ["recombinant human activated protein C", "low dose of corticosteroids", "Xigris"], "intervention_types": ["DRUG"], "allocation": null}
--- stopped=True  tokens=60
{"conditions": ["acute leukemia", "high-risk acute leukemia patients"], "interventions": ["Sheng Yu Tang (聖愈湯, SYT)", "standard-care treatment"], "intervention_types": ["DRUG"], "allocation": null}
--- stopped=True  tokens=39
{"conditions": ["critically ill patients"], "interventions": ["Dexcom G6 glucose monitoring system"], "intervention_types": ["DEVICE"], "allocation": null}
--- stopped=True  tokens=45
{"conditions": ["Type 2 Diabetes"], "interventions": ["Nurse-led empowerment training"], "intervention_types": ["BEHAVIORAL"], "allocati

## Save model and adapter

In [11]:
import os

OUT = os.path.expanduser("~/smol_llm/models/sft-cpt")   # "sft-cpt" for the CPT arm

model.save_pretrained(f"{OUT}/adapter")                  # LoRA weights only, for DPO

merged = model.merge_and_unload().to(torch.bfloat16)
merged.save_pretrained(f"{OUT}/merged")                  # full model for vLLM, no tokenizer

print(os.listdir(f"{OUT}/adapter"), os.listdir(f"{OUT}/merged"))

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.63s/it]

['README.md', 'adapter_model.safetensors', 'adapter_config.json'] ['generation_config.json', 'model.safetensors', 'config.json']
